In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv('../data/ecommerce_pricing.csv')

In [3]:
X = df.drop(columns=['product_id', 'optimal_price'])
y = df['optimal_price']

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
X_train.shape,X_test.shape,y_train.shape,y_test.shape

((9600, 8), (2400, 8), (9600,), (2400,))

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [7]:
num_features = ['base_cost', 'competitor_price', 'inventory_level', 'demand_score', 'customer_rating', 'is_weekend', 'shipping_days']
cat_features = ['product_category']

In [8]:
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

In [9]:
cat_pipeline = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

In [10]:
preprocessor = ColumnTransformer(transformers=[
    ('num_pipe', num_pipeline, num_features),
    ('cat_pipe', cat_pipeline, cat_features)
])

In [11]:
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

In [12]:
X_train_transformed.shape,X_test_transformed.shape

((9600, 12), (2400, 12))

In [15]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [16]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, random_state=42),
    "CatBoost": CatBoostRegressor(verbose=0, random_state=42)
}

In [17]:
results_list = []
for model_name, model in models.items():
    model.fit(X_train_transformed, y_train)
    y_pred = model.predict(X_test_transformed)
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    
    results_list.append({
        "Model": model_name,
        "R2 Score": round(r2, 5),
        "RMSE": round(rmse, 2),
        "MAE": round(mae, 2)
    })

In [18]:
results_df = pd.DataFrame(results_list).sort_values(by="R2 Score", ascending=False)
results_df.reset_index(drop=True, inplace=True)
results_df

,Model,R2 Score,RMSE,MAE
0,CatBoost,0.99873,6.28,3.84
1,XGBoost,0.99802,7.84,4.74
2,Gradient Boosting,0.99754,8.74,5.49
3,Random Forest,0.99693,9.76,5.40
4,Ridge,0.99420,13.42,8.25
5,Linear Regression,0.99420,13.42,8.26
